# 01 — Cleaning & Risk Label Construction

Goal: clean the raw collected data, and construct a defensible "at risk" 
label — since GitHub doesn't hand us one, we have to build it from the 
signals we collected and justify the definition.

In [12]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_json('../data/repo_health_raw.json')
print(df.shape)
df.head()

(499, 21)


,repo,stars,forks,language,license,is_archived,age_days,days_since_last_push,days_since_last_commit,commits_per_month_90d,commits_per_month_365d,distinct_recent_commit_authors,total_mentionable_users,open_issues_total,closed_issues_total,median_issue_resolution_days_recent,open_prs_total,merged_prs_total,median_pr_merge_days_recent,pr_merge_rate_recent,star_tier
0,codecrafters-io/build-your-own-x,544722,51316,Markdown,NaN,False,3038,49,49.0,1.67,1.92,38,131,275,673,0.0,365,157,NaN,0.00,mega
1,sindresorhus/awesome,502221,36685,NaN,cc0-1.0,False,4435,0,0.0,2.33,3.00,68,621,16,352,0.0,89,702,0.0,0.10,mega
2,public-apis/public-apis,474426,52388,Python,mit,False,3817,0,0.0,151.67,59.00,37,1487,35,911,4.0,1813,2190,1.0,0.32,mega
3,freeCodeCamp/freeCodeCamp,454881,46100,TypeScript,bsd-3-clause,False,4269,0,0.0,269.67,292.08,39,6226,128,21987,0.0,92,29249,0.0,0.42,mega
4,EbookFoundation/free-programming-books,395797,66724,Python,cc-by-4.0,False,4709,1,1.0,20.00,51.17,78,3204,39,1258,2.0,44,7431,1.0,0.46,mega


In [13]:
df.isnull().sum().sort_values(ascending=False)

median_pr_merge_days_recent            84
license                                81
language                               60
median_issue_resolution_days_recent    56
pr_merge_rate_recent                   33
days_since_last_commit                  1
stars                                   0
repo                                    0
forks                                   0
commits_per_month_90d                   0
is_archived                             0
age_days                                0
days_since_last_push                    0
total_mentionable_users                 0
distinct_recent_commit_authors          0
commits_per_month_365d                  0
open_issues_total                       0
open_prs_total                          0
closed_issues_total                     0
merged_prs_total                        0
star_tier                               0
dtype: int64

In [14]:
df[df['days_since_last_commit'].isnull()]

,repo,stars,forks,language,license,is_archived,age_days,days_since_last_push,days_since_last_commit,commits_per_month_90d,commits_per_month_365d,distinct_recent_commit_authors,total_mentionable_users,open_issues_total,closed_issues_total,median_issue_resolution_days_recent,open_prs_total,merged_prs_total,median_pr_merge_days_recent,pr_merge_rate_recent,star_tier
456,niie-tuong/ebookenglish,100,0,NaN,NaN,False,1100,1100,NaN,0.0,0.0,0,1,0,0,NaN,0,0,NaN,NaN,tiny


In [15]:
# Drop repos with no commit history at all - these are empty/placeholder
# repos, not real ongoing projects, so "days since last commit" is
# undefined rather than missing. Confirmed only 1 such case.
before = len(df)
df = df[df['days_since_last_commit'].notnull()].copy()
print(f"Dropped {before - len(df)} row(s) with no commit history. New shape: {df.shape}")

Dropped 1 row(s) with no commit history. New shape: (498, 21)


In [16]:
df['language'] = df['language'].fillna('None/Not detected')
df['license'] = df['license'].fillna('No license')

df[['language', 'license']].isnull().sum()

language    0
license     0
dtype: int64

In [17]:
null_pr_rate = df[df['pr_merge_rate_recent'].isnull()]
print("Rows with null pr_merge_rate_recent:", len(null_pr_rate))
print(null_pr_rate[['repo', 'open_prs_total', 'merged_prs_total', 'star_tier']].describe(include='all'))
print()
print(null_pr_rate[['open_prs_total', 'merged_prs_total']].sum())

Rows with null pr_merge_rate_recent: 32
                                         repo  open_prs_total  merged_prs_total star_tier
count                                      32            32.0              32.0        32
unique                                     32             NaN               NaN         5
top     awesome-selfhosted/awesome-selfhosted             NaN               NaN      tiny
freq                                        1             NaN               NaN        20
mean                                      NaN             0.0               0.0       NaN
std                                       NaN             0.0               0.0       NaN
min                                       NaN             0.0               0.0       NaN
25%                                       NaN             0.0               0.0       NaN
50%                                       NaN             0.0               0.0       NaN
75%                                       NaN             0.

In [18]:
df['has_recent_pr_activity'] = df['open_prs_total'] + df['merged_prs_total'] > 0
df['has_recent_issue_activity'] = df['open_issues_total'] + df['closed_issues_total'] > 0

print(df['has_recent_pr_activity'].value_counts())
print(df['has_recent_issue_activity'].value_counts())

has_recent_pr_activity
True     460
False     38
Name: count, dtype: int64
has_recent_issue_activity
True     454
False     44
Name: count, dtype: int64


In [19]:
for col in ['median_issue_resolution_days_recent', 'median_pr_merge_days_recent', 'pr_merge_rate_recent']:
    df[col] = df[col].fillna(-1)

df.isnull().sum().sum()  # should be 0 now

np.int64(0)

In [20]:
signal_cols = ['days_since_last_commit', 'commits_per_month_90d', 
               'distinct_recent_commit_authors', 'open_issues_total']

df.groupby('is_archived')[signal_cols].median()

,days_since_last_commit,commits_per_month_90d,distinct_recent_commit_authors,open_issues_total
is_archived,,,,
False,50.0,0.67,9.0,30.0
True,2051.0,0.00,5.0,6.0


In [21]:
df['days_since_last_commit'].quantile([0.1, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95, 0.99])

0.10       0.00
0.25       1.00
0.50      80.00
0.60     287.80
0.70     691.20
0.75    1014.00
0.80    1369.60
0.90    2201.60
0.95    2898.90
0.99    4061.06
Name: days_since_last_commit, dtype: float64

In [24]:
def compute_risk_score(row):
    if row['is_archived']:
        return 100

    score = 0
    if row['days_since_last_commit'] > 730:
        score += 50
    elif row['days_since_last_commit'] > 365:
        score += 30
    elif row['days_since_last_commit'] > 180:
        score += 10

    if row['commits_per_month_90d'] == 0:
        score += 25
    elif row['commits_per_month_90d'] < 1:
        score += 10

    # Bus factor: only penalize low contributor count when it's NOT
    # contradicted by clear recent velocity. distinct_recent_commit_authors
    # can undercount when commit authors' emails aren't linked to a GitHub
    # account (bots, unlinked emails) - a repo with 5+ commits/month is
    # demonstrably active regardless of what that count says.
    if row['distinct_recent_commit_authors'] <= 1 and row['commits_per_month_90d'] < 5:
        score += 25

    return min(score, 100)

df['risk_score'] = df.apply(compute_risk_score, axis=1)
df['risk_category'] = df['risk_score'].apply(risk_category)
df['risk_score'].describe()

count    498.000000
mean      35.712851
std       39.352709
min        0.000000
25%        0.000000
50%       10.000000
75%       75.000000
max      100.000000
Name: risk_score, dtype: float64

In [23]:
def risk_category(score):
    if score >= 75:
        return 'High'
    elif score >= 25:
        return 'Medium'
    else:
        return 'Low'

df['risk_category'] = df['risk_score'].apply(risk_category)
print(df['risk_category'].value_counts())
print()
# Sanity check: every archived repo should land in High risk
print("Archived repos NOT flagged High risk (should be 0):")
print(df[(df['is_archived']) & (df['risk_category'] != 'High')][['repo', 'risk_score', 'risk_category']])

risk_category
Low       250
High      162
Medium     86
Name: count, dtype: int64

Archived repos NOT flagged High risk (should be 0):
Empty DataFrame
Columns: [repo, risk_score, risk_category]
Index: []


In [25]:
print("=== Sample HIGH risk repos ===")
print(df[df['risk_category']=='High'][['repo','stars','days_since_last_commit','commits_per_month_90d','distinct_recent_commit_authors','is_archived']].sample(8, random_state=1))

print("\n=== Sample LOW risk repos (high star, active) ===")
print(df[(df['risk_category']=='Low') & (df['stars']>10000)][['repo','stars','days_since_last_commit','commits_per_month_90d','distinct_recent_commit_authors']].sample(5, random_state=1))

=== Sample HIGH risk repos ===
                                         repo  stars  days_since_last_commit  commits_per_month_90d  distinct_recent_commit_authors  is_archived
302          arnesson/cordova-plugin-firebase   1000                   948.0                    0.0                              10        False
308                        davidedc/Algebrite   1000                  1625.0                    0.0                               4        False
423                         v4rl-ucy/decoSLAM    100                  1197.0                    0.0                               1        False
374                       hanks-zyh/SmallBang    999                  3204.0                    0.0                               2        False
494                 DirtyHarryLYL/HAKE-Action    100                  1336.0                    0.0                               1        False
200                    petehunt/webpack-howto  10000                  3795.0                    0.0

In [26]:
df.to_csv('../data/clean_labeled_repos.csv', index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns to data/clean_labeled_repos.csv")

Saved 498 rows, 25 columns to data/clean_labeled_repos.csv


## Cleaning & Labeling Summary

| Issue | Rows affected | Decision |
|---|---|---|
| No commit history at all (empty/placeholder repo) | 1 | Dropped |
| Missing `language` | 60 | Filled with `"None/Not detected"` |
| Missing `license` | 81 | Filled with `"No license"` |
| Missing PR/issue timing stats (no PRs/issues to measure) | 32-84 depending on column | Filled with sentinel `-1`, flagged via `has_recent_pr_activity` / `has_recent_issue_activity` |

**Risk label:** built as a composite `risk_score` (0-100) rather than a single 
threshold, combining dormancy, recent commit velocity, and contributor count 
(bus factor), with `is_archived` repos automatically scored as maximum risk 
since that's the one piece of ground truth in the data. Validated against:
archived repos all land in High risk (0 exceptions), and spot-checks of 
known-dead vs. known-active repos (e.g. `webpack-howto` vs. `flutter`) match 
intuition.

**Output:** `data/clean_labeled_repos.csv` — 498 rows, ready for SQL analysis and EDA.